# Orchestral Hello World

In [1]:
import sys
print(sys.version)
assert sys.version_info[:2] == (3, 13), "Run this notebook with Python 3.13"

3.13.12 | packaged by Anaconda, Inc. | (main, Feb 24 2026, 16:07:20) [Clang 20.1.8 ]


In [2]:
!pip install orchestral-ai -q

In [3]:
import os
import re
import time
from dotenv import load_dotenv

from orchestral import Agent, define_tool
from orchestral.llm import Gemini
from orchestral.llm.google.pricing_model import pricing_model
from google.api_core import exceptions as google_exceptions
from google.genai.errors import ClientError as GeminiClientError

load_dotenv()

# Orchestral 1.3.0 has an older Gemini model allowlist.
if "gemini-2.5-flash" not in pricing_model.rates:
    pricing_model.rates["gemini-2.5-flash"] = pricing_model.rates["gemini-2.0-flash-exp"]

# Compatibility patch for google-genai where system_instruction/tools must be in config.
def _build_sdk_config(self, formatted_input, kwargs):
    system_instruction = formatted_input.get("system_instruction")
    generation_config = kwargs.pop("generation_config", None) or {}
    user_config = kwargs.pop("config", None)
    merged = {}

    if isinstance(user_config, dict):
        merged.update(user_config)
    elif user_config is not None and hasattr(user_config, "model_dump"):
        merged.update(user_config.model_dump(exclude_none=True))

    if generation_config:
        merged["generation_config"] = generation_config
    if system_instruction:
        merged["system_instruction"] = system_instruction
    if self.tool_schemas:
        merged["tools"] = self.tool_schemas

    return merged or None

def _gemini_call_api(self, formatted_input, use_prompt_cache=False, **kwargs):
    config = self._build_sdk_config(formatted_input, kwargs)
    try:
        return self.client.generate_content(
            model=self.model,
            contents=formatted_input["messages"],
            config=config,
            **kwargs,
        )
    except google_exceptions.ResourceExhausted as e:
        self._handle_quota_error(e)

def _gemini_call_streaming_api(self, formatted_input, use_prompt_cache=False, **kwargs):
    config = self._build_sdk_config(formatted_input, kwargs)
    try:
        return self.client.generate_content_stream(
            model=self.model,
            contents=formatted_input["messages"],
            config=config,
            **kwargs,
        )
    except google_exceptions.ResourceExhausted as e:
        self._handle_quota_error(e)

Gemini._build_sdk_config = _build_sdk_config
Gemini.call_api = _gemini_call_api
Gemini.call_streaming_api = _gemini_call_streaming_api

MODEL_NAME = "gemini-2.5-flash"

## Define tool

In [4]:
@define_tool()
def hilbert_space_dimension(n_qubits: int) -> dict:
    """
    Compute the Hilbert-space dimension of an n-qubit quantum system.

    Use this tool whenever the user asks for:
    - the Hilbert-space dimension of n qubits
    - the size of the state space of an n-qubit system
    - the number of amplitudes required to represent n qubits

    Args:
        n_qubits: Number of qubits. Must be a nonnegative integer.

    Returns:
        A dictionary with:
        - n_qubits: input number of qubits
        - dimension: 2 ** n_qubits
    """
    if n_qubits < 0:
        raise ValueError("n_qubits must be nonnegative")

    result = {"n_qubits": n_qubits, "dimension": 2 ** n_qubits}
    TOOL_CALL_LOG.append(result)
    return result

In [5]:
agent = Agent(
    llm=Gemini(model=MODEL_NAME),
    tools=[hilbert_space_dimension],
    system_prompt=(
        "You are a quantum-computing assistant. "
        "When asked for the Hilbert-space dimension or state-space size "
        "of an n-qubit system, use the available tool."
    ),
)

In [6]:
TOOL_CALL_LOG = []
TOOL_CALL_LOG.clear()
# response = agent.run("What is the Hilbert space dimension of a 4-qubit system?")
response = agent.run("How many amplitudes are needed to represent a 3-qubit state?")
print(response.text)
print("Tool log:", TOOL_CALL_LOG)

You need 8 amplitudes to represent a 3-qubit state.
Tool log: [{'n_qubits': 3, 'dimension': 8}]


## Test reliability

In [7]:
# prompts = [
#     "What is the Hilbert space dimension of a 2-qubit system?",
#     "How large is the state space for 5 qubits?",
#     "How many amplitudes are needed for 7 qubits?",
#     "Compute the Hilbert-space size for 1 qubit."
# ]

# def run_with_quota_retry(prompt, max_retries=4):
#     for attempt in range(max_retries + 1):
#         try:
#             return agent.run(prompt)
#         except GeminiClientError as e:
#             message = str(e)
#             is_quota = "RESOURCE_EXHAUSTED" in message or "429" in message
#             if not is_quota or attempt == max_retries:
#                 raise

#             retry_match = re.search(r"retry in ([0-9.]+)s", message.lower())
#             wait_seconds = int(float(retry_match.group(1))) + 1 if retry_match else 20
#             print(f"Rate limited. Waiting {wait_seconds}s before retry...")
#             time.sleep(wait_seconds)

# TOOL_CALL_LOG.clear()
# for p in prompts:
#     response = run_with_quota_retry(p)
#     print(p)
#     print(response.text)
#     print()

# print("Number of tool calls:", len(TOOL_CALL_LOG))
# print("Tool log:", TOOL_CALL_LOG)